In [1]:
import os
import json
import torch
import bisect

import numpy as np
import polars as pl

from torch import nn

from math import ceil
from pathlib import Path
from collections import defaultdict
from datasets import load_from_disk
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from typing import Dict, Iterable, List, Optional, Any, Union, Literal, Tuple, Callable

In [2]:
class Tokenizer:
    def __init__(
        self,
        codes_parquet_fp: str,
        special_tokens: Optional[Iterable[str]] = None,
        force_special_ids: bool = True,  # pin [PAD]=0 etc.
    ):
        if special_tokens is None:
            special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]


        df_codes = pl.read_parquet(str(codes_parquet_fp), columns=["code"])
        base_codes = df_codes.get_column("code").to_list()
        seen = set()
        unique_codes = []
        for c in base_codes:
            if c not in seen:
                unique_codes.append(c)
                seen.add(c)

        vocab_list: List[str] = []
        special_tokens = list(special_tokens)

        if force_special_ids:
            for tok in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]:
                if tok in special_tokens and tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
                elif tok in special_tokens and tok in seen:

                    vocab_list.append(tok)
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
        else:
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)


        vocab_list.extend(unique_codes)


        self.id2code: List[str] = vocab_list
        self.code2id: Dict[str, int] = {tok: idx for idx, tok in enumerate(vocab_list)}
        self.vocab_size: int = len(self.id2code)

        self.pad_token  = "[PAD]" if "[PAD]" in self.code2id else None
        self.mask_token = "[MASK]" if "[MASK]" in self.code2id else None
        self.cls_token  = "[CLS]" if "[CLS]" in self.code2id else None
        self.unk_token  = "[UNK]" if "[UNK]" in self.code2id else None

        self.pad_id  = self.code2id[self.pad_token]  if self.pad_token  else 0
        self.mask_id = self.code2id[self.mask_token] if self.mask_token else None
        self.cls_id  = self.code2id[self.cls_token]  if self.cls_token  else None
        self.unk_id  = self.code2id[self.unk_token]  if self.unk_token  else None


        type_set = set()
        for tok in self.id2code:
            prefix = tok.split("//", 1)[0]
            type_set.add(prefix)

        types_sorted = sorted(t for t in type_set if t not in ("[PAD]",))
        self.type2id: Dict[str, int] = {"[PAD]": 0}
        next_id = 1
        for sp in ["[MASK]", "[CLS]", "[UNK]"]:
            if sp in type_set:
                self.type2id[sp] = next_id; next_id += 1
        for t in types_sorted:
            if t not in self.type2id:
                self.type2id[t] = next_id
                next_id += 1

        self._code2id_df = pl.DataFrame({"code": self.id2code,
                                         "input_id": list(range(self.vocab_size))}) \
                               .with_columns(pl.col("code").cast(pl.Categorical))
        self._type2id_df = pl.DataFrame({"code_type": list(self.type2id.keys()),
                                         "type_id":   list(self.type2id.values())}) \
                               .with_columns(pl.col("code_type").cast(pl.Categorical))


    def encode(self, codes: Iterable[str]) -> List[int]:
        get = self.code2id.get
        if self.unk_id is not None:
            fallback = self.unk_id
        else:
            fallback = self.pad_id if self.pad_id is not None else 0
        return [get(c, fallback) for c in codes]

    def decode(self, ids: Iterable[int]) -> List[str]:
        out = []
        for i in ids:
            if 0 <= i < self.vocab_size:
                out.append(self.id2code[i])
            else:
                out.append(self.unk_token or "[UNK]")
        return out

    def save(self, path: str) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        obj = {
            "id2code": self.id2code,
            "special_tokens": [t for t in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"] if t in self.code2id],
            "type2id": self.type2id,
            "pad_id": self.pad_id,
            "mask_id": self.mask_id,
            "cls_id": self.cls_id,
            "unk_id": self.unk_id,
        }
        with open(path, "w") as f:
            json.dump(obj, f, indent=2)

    @classmethod
    def load(cls, path: str) -> "Tokenizer":
        path = Path(path)
        with open(path) as f:
            obj = json.load(f)

        tok = cls.__new__(cls) 

        tok.id2code = obj["id2code"]
        tok.code2id = {tok_: i for i, tok_ in enumerate(tok.id2code)}
        tok.vocab_size = len(tok.id2code)

        tok.special_tokens = obj.get("special_tokens", [])
        tok.type2id = obj.get("type2id", {})

        tok.pad_token  = "[PAD]" if "[PAD]" in tok.code2id else None
        tok.mask_token = "[MASK]" if "[MASK]" in tok.code2id else None
        tok.cls_token  = "[CLS]" if "[CLS]" in tok.code2id else None
        tok.unk_token  = "[UNK]" if "[UNK]" in tok.code2id else None

        tok.pad_id  = obj.get("pad_id", tok.code2id.get("[PAD]", 0))
        tok.mask_id = obj.get("mask_id", tok.code2id.get("[MASK]")) if "[MASK]" in tok.code2id else None
        tok.cls_id  = obj.get("cls_id", tok.code2id.get("[CLS]"))   if "[CLS]" in tok.code2id else None
        tok.unk_id  = obj.get("unk_id", tok.code2id.get("[UNK]"))   if "[UNK]" in tok.code2id else None

        # Rebuild Polars lookup frames
        tok._code2id_df = pl.DataFrame({"code": tok.id2code,
                                        "input_id": list(range(tok.vocab_size))}) \
                              .with_columns(pl.col("code").cast(pl.Categorical))
        tok._type2id_df = pl.DataFrame({"code_type": list(tok.type2id.keys()),
                                        "type_id":   list(tok.type2id.values())}) \
                              .with_columns(pl.col("code_type").cast(pl.Categorical))
        return tok

    @property
    def code2id_df(self) -> pl.DataFrame:
        return self._code2id_df

    @property
    def type2id_df(self) -> pl.DataFrame:
        return self._type2id_df

In [3]:
class SequencesGenerator:
    def __init__(
        self,
        tokenizer_path: str,
        chunk_length: int = 1024,
        overlap: int = 128,
        return_numeric: bool = False,
        return_text: bool = False,
        return_time: bool = False,
        return_ids: bool = False,
    ):

        self.tokenizer = Tokenizer.load(tokenizer_path)
        self.chunk_length = chunk_length
        self.overlap = overlap
        self.return_numeric = return_numeric
        self.return_text = return_text
        self.return_time = return_time
        self.return_ids = return_ids

    def encode_sequence(
        self,
        timeline: pl.DataFrame,
        max_length: Optional[int] = None,
        pad_to_max: bool = False,
        truncation: Literal["head", "tail"] = "tail",
        add_cls: bool = False,
    ) -> Dict[str, Union[List[int], List[float], List[str]]]:
        """
        Vectorized build of:
          input_ids, attention_mask, visit_ids, stage_ids, type_ids
          + optional numeric/text streams (+ masks)
        """
        df = timeline
        
        if "seq_id" in df.columns:

            uniq = df.select(pl.col("seq_id")).unique(maintain_order=True)
            uniq = uniq.with_row_count(name="visit_ids_raw")  # 0..K-1
            df = df.join(uniq, on="seq_id", how="left").with_columns(
                (pl.col("visit_ids_raw") ).alias("visit_id").fill_null(0)
            ).drop("visit_ids_raw")
        else:
            df = df.with_columns(pl.lit(0).alias("visit_id"))

        stage_cols = ["out_id", "er_id", "hadm_id", "icustay_id"]
        present_stages = [c for c in stage_cols if c in df.columns]
        if present_stages:

            expr = pl.lit(0)
            for i, col in enumerate(present_stages, start=1):
                expr = pl.when(expr.eq(0) & pl.col(col).is_not_null()).then(i).otherwise(expr)
            df = df.with_columns(expr.alias("stage_id"))
        else:
            df = df.with_columns(pl.lit(0).alias("stage_id"))

        df = df.join(
            self.tokenizer.type2id_df,
            on=pl.col("code_type").cast(pl.Categorical),
            how="left",
        ).with_columns(pl.col("type_id").fill_null(0))

        df = df.join(
            self.tokenizer.code2id_df,
            on=pl.col("code").cast(pl.Categorical),
            how="left",
        )


        unk_id = self.tokenizer.unk_id if self.tokenizer.unk_id is not None else self.tokenizer.pad_id or 0
        df = df.with_columns(pl.col("input_id").fill_null(unk_id))

        df = df.with_columns(
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("visit_id"))
              .alias("visit_id"),
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("stage_id"))
              .alias("stage_id"),
        )


        if add_cls and (self.tokenizer.cls_token is not None):
            cls_row = {
                "code": self.tokenizer.cls_token,
                "code_type": "[CLS]",
                "visit_id": 0,
                "stage_id": 0,
                "type_id": self.tokenizer.type2id.get("[CLS]", 0),
                "input_id": self.tokenizer.cls_id,
            }

            if self.return_numeric:
                cls_row["numeric_value"] = None
            if self.return_text:
                cls_row["text_value"] = None

            df = pl.concat([pl.DataFrame([cls_row]), df], how="vertical_relaxed")


        input_ids = df.get_column("input_id").cast(pl.Int64).to_list()
        type_ids = df.get_column("type_id").cast(pl.Int64).to_list()
        visit_ids = df.get_column("visit_id").cast(pl.Int64).to_list()
        stage_ids = df.get_column("stage_id").cast(pl.Int64).to_list()
        attention_mask = [1] * len(input_ids)


        value_payload = self._build_value_streams(
            df=df,
            max_length=max_length,
            pad_to_max=pad_to_max,
            truncation=truncation,
        )

        # --- truncate/pad core streams in one go ---
        input_ids      = self._truncate(input_ids,      max_length, truncation)
        type_ids       = self._truncate(type_ids,       max_length, truncation)
        visit_ids      = self._truncate(visit_ids,      max_length, truncation)
        stage_ids      = self._truncate(stage_ids,      max_length, truncation)
        attention_mask = [1] * len(input_ids)

        if pad_to_max and max_length is not None and len(input_ids) < max_length:
            pad_len = max_length - len(input_ids)
            pad_id = self.tokenizer.pad_id if self.tokenizer.pad_id is not None else 0
            input_ids      = input_ids + [pad_id] * pad_len
            type_ids       = type_ids + [0] * pad_len
            visit_ids      = visit_ids + [0] * pad_len
            stage_ids      = stage_ids + [0] * pad_len
            attention_mask = attention_mask + [0] * pad_len

        out = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "visit_ids": visit_ids,
            "stage_ids": stage_ids,
            "type_ids": type_ids,
        }
        out.update(value_payload)
        return out

    def get_overlapped_chunks(
        self,
        timeline: Dict[str, Iterable],
        chunk_length: Optional[int] = None,
        overlap: Optional[int] = None,
        add_cls_per_chunk: bool = True,
    ) -> List[Dict[str, List[Any]]]:
        """
        Sliding-window chunking with optional [CLS] per chunk and padding.
        """
        if chunk_length is None or overlap is None:
            chunk_length = self.chunk_length
            overlap = self.overlap

        fields = ["input_ids", "attention_mask", "visit_ids", "stage_ids", "type_ids"]
        for extra in ("numeric_values", "numeric_mask", "text_values", "text_mask", 'time_diff'):
            if extra in timeline and extra not in fields:
                fields.append(extra)

        n = len(timeline["input_ids"])
        payload = chunk_length - (1 if add_cls_per_chunk else 0)
        step = max(1, payload - overlap)

        num_chunks = 1 if n <= payload else ceil((n - payload) / step) + 1
        starts = [i * step for i in range(num_chunks)]

        cls_defaults = {
            "input_ids": self.tokenizer.cls_id if self.tokenizer.cls_id is not None else (self.tokenizer.pad_id or 0),
            "attention_mask": 1,
            "visit_ids": 0,
            "stage_ids": 0,
            "type_ids": self.tokenizer.type2id.get("[CLS]", 0),
            "numeric_values": 0.0,
            "numeric_mask": 0,
            "text_values": "",
            "text_mask": 0,
            "time_diff":0.0}

        chunks = []
        for start in starts:
            end = min(n, start + payload)
            sliced = {k: list(timeline[k][start:end]) for k in fields if k in timeline}

            if "attention_mask" in sliced:
                sliced["attention_mask"] = [1] * len(sliced["input_ids"])

            if add_cls_per_chunk:
                for k in list(sliced.keys()):
                    sliced[k] = [cls_defaults[k]] + sliced[k]

            cur_len = len(sliced["input_ids"])
            if cur_len < chunk_length:
                pad_len = chunk_length - cur_len
                for k in list(sliced.keys()):
                    sliced[k] = self._pad_list(sliced[k], pad_len, 0)

            chunks.append(sliced)
        return chunks


    def _build_value_streams(
        self,
        df: pl.DataFrame,
        max_length: Optional[int],
        pad_to_max: bool,
        truncation: Literal["head", "tail"],
    ) -> Dict[str, List[Any]]:
        out: Dict[str, List[Any]] = {}
        # Numeric stream
        if self.return_numeric:
            if "numeric_value" in df.columns:
                vals = df.get_column("numeric_value").to_list()
            else:
                vals = [None] * df.height
            num_mask = [1 if (v is not None) else 0 for v in vals]
            vals = [0.0 if v is None else float(v) for v in vals]

            vals = self._truncate(vals, max_length, truncation)
            num_mask = self._truncate(num_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(vals) < max_length:
                pad_len = max_length - len(vals)
                vals += [0.0] * pad_len
                num_mask += [0] * pad_len

            out["numeric_values"] = vals
            out["numeric_mask"] = num_mask

        # Text stream
        if self.return_text:
            if "text_value" in df.columns:
                txt = df.get_column("text_value").to_list()
            else:
                txt = [None] * df.height
            txt = [("" if (t is None or str(t) == "___") else str(t)) for t in txt]
            txt_mask = [1 if (t != "") else 0 for t in txt]

            txt = self._truncate(txt, max_length, truncation)
            txt_mask = self._truncate(txt_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(txt) < max_length:
                pad_len = max_length - len(txt)
                txt += [""] * pad_len
                txt_mask += [0] * pad_len

            out["text_values"] = txt
            out["text_mask"] = txt_mask

            
        if self.return_time:
            if "time_diff" in df.columns:
                df = df.with_columns(pl.col(['time_diff'])).fill_null(0.0)
                time_diff = df.get_column("time_diff").to_list()
                time_diff = self._scale_time_deltas(time_diff)
                time_stamp = df.get_column("time").to_list()
            else:
                time_diff = [None] * df.height
                time_stamp = [None] * df.height


            if pad_to_max and max_length is not None and len(time_diff) < max_length:
                pad_len = max_length - len(time_diff)
                time_diff += [0] * pad_len
                time_stamp += [0] * pad_len


            out["time_diff"] = time_diff
            out["time_stamp"] = time_stamp
            
        if self.return_ids:
            if "seq_id" in df.columns:
                
                seq_id = df.get_column("seq_id").cast(pl.Int32).to_list()
                out_id = df.get_column("out_id").cast(pl.Int32).to_list()
                er_id =  df.get_column("er_id").cast(pl.Int32).to_list()
                hadm_id = df.get_column("hadm_id").cast(pl.Int32).to_list()
                icustay_id = df.get_column("hadm_id").cast(pl.Int32).to_list()
            else:
                seq_id = [None] * df.height
                out_id = [None] * df.height
                er_id =  [None] * df.height
                hadm_id = [None] * df.height
                icustay_id = [None] * df.height

            if pad_to_max and max_length is not None and len(time_diff) < max_length:
                pad_len = max_length - len(time_diff)
                seq_id += [0] * pad_len
                out_id += [0] * pad_len
                er_id += [0] * pad_len
                hadm_id += [0] * pad_len
                icustay_id += [0] * pad_len

            out["seq_id"] = seq_id
            out["out_id"] = out_id
            out["er_id"] = er_id
            out["hadm_id"] = hadm_id
            out["icustay_id"] = icustay_id

        return out
    
    def _scale_time_deltas(self, deltas_list):
        deltas = np.asarray(deltas_list, dtype=float)
        compressed = np.log1p(deltas)              
        scaled = compressed / np.log(5328.93125)         
        return scaled.tolist()

    @staticmethod
    def _truncate(seq: List[Any], max_length: Optional[int], truncation: str) -> List[Any]:
        if max_length is None or len(seq) <= max_length:
            return seq
        return seq[-max_length:] if truncation == "head" else seq[:max_length]

    @staticmethod
    def _pad_list(lst: List[Any], pad_len: int, pad_value: Any) -> List[Any]:
        if pad_len <= 0:
            return lst
        return lst + [pad_value] * pad_len

In [4]:
class EHRPretrainDataset(Dataset):
    def __init__(self,
                 dataset_path: str,
                 data_idx_path: str,
                 seq_generator: SequencesGenerator,
                 needed_cols: list = ['subject_id', 'input_ids', 'attention_mask', 'visit_ids', 'stage_ids', 'type_ids'],
                 split: str = 'all') -> None:
        
        hf_dataset = load_from_disk(dataset_path)
        self.hf_dataset = hf_dataset.flatten_indices().select_columns(needed_cols) \
                                      .with_format("numpy", columns=needed_cols, output_all_columns=False)
        
        self.seq_generator = seq_generator
        
        sids = self.hf_dataset["subject_id"]        
        self.index = defaultdict(list)
        for i, sid in enumerate(sids):
            self.index[sid].append(i)
        
        data_idx =  pl.scan_parquet(data_idx_path).collect()
        splits = {'all': data_idx,
                  'train':data_idx.filter(pl.col('split') == 'train'),
                  'val':  data_idx.filter(pl.col('split') == 'val')}

        self.data_idx, self.cum, self.subj = self._get_chunks_count(data_idx=splits[split],
                                                                    chunk_length=self.seq_generator.chunk_length,
                                                                    overlap=self.seq_generator.overlap)
        
        
    def __len__(self) -> int:
        return self.cum[-1]


    
    def __getitem__(self,
                    idx: int):
        

        subject_id, chunk_id = self._get_chunk_at_idx(idx=idx,
                                                      cumm_sum=self.cum,
                                                      subjects=self.subj)
        

        timeline_encoded = self.hf_dataset.select(self.index[subject_id])[0]
        chunks = self.seq_generator.get_overlapped_chunks(timeline= timeline_encoded,
                                                          chunk_length= self.seq_generator.chunk_length,
                                                          overlap=self.seq_generator.overlap)
        

        return chunks[chunk_id]
    

    def _build_dataset_index(self,
                             data_path, 
                             subject_col="subject_id") -> pl.DataFrame:
        pieces = []
        for p in os.listdir(data_path):
            df = (pl.scan_parquet(os.path.join(data_path,p)).select(subject_col).collect()
                    .group_by(subject_col)
                    .len()
                    .rename({"len": "n_events"})
                 )
            df = df.with_columns(pl.lit(str(p)).alias("shard"))  # optional
            pieces.append(df)


        df = (pl.concat(pieces, how="vertical")
                  .group_by([subject_col, "shard"])
                  .agg(pl.col("n_events").sum())
                  .rename({subject_col:"subject_id"})).sort('subject_id')

        df = df.filter(pl.col('n_events') >3)

        return df
    
    
    def _read_timeline(self,
                       subject_id:int) -> pl.DataFrame:
        
        shard = self.data_idx.filter(pl.col('subject_id') == subject_id)['shard'][0]

        data = pl.scan_parquet(os.path.join(self.data_path,shard),parallel='auto').select(
                                            ['subject_id','seq_id','out_id','er_id','hadm_id', 
                                             'icustay_id','time','code','numeric_value','code_type',
                                             'text_value']).filter(
                                              pl.col('subject_id') == subject_id).collect()
        return data

    
    def _get_chunks_count(self,
                          data_idx: pl.DataFrame,
                          chunk_length: int,
                          overlap: int):
        payload  = chunk_length - 1
        step     = payload - overlap

        data_idx = data_idx.with_columns(
            pl.col("n_events")
              .map_elements(lambda n: 1 if n<=payload else ceil((n-payload)/step)+1,return_dtype=pl.Int32)
              .alias("n_chunks")
        )
        data_idx = data_idx.with_columns(
            pl.col("n_chunks").cum_sum().alias("cum_chunks")
        )

        cum  = data_idx["cum_chunks"]   
        subj = data_idx["subject_id"]
        shards = data_idx['shard']
        return data_idx, cum, subj

    def _get_chunk_at_idx(self,
                          cumm_sum: list,
                          subjects: list,
                          idx: int) -> Tuple[int,int,str]:
        i = bisect.bisect_right(cumm_sum, idx)
        left = cumm_sum[i-1] if i > 0 else 0
        return subjects[i], idx - left

In [5]:
class MLMDataCollator:
    def __init__(
        self,
        tokenizer,
        protected_tokens: List[str],
        mask_prob: float = 0.15,
        replace_prob: float = 0.80,
        random_prob: float = 0.10,
    ) -> None:

        self.tokenizer = tokenizer
        self.mask_prob = mask_prob
        self.replace_prob = replace_prob
        self.random_prob = random_prob
        self.protected_ids = self._build_protected_ids(protected_tokens)
        if tokenizer.mask_id is None:
            raise ValueError("Tokenizer must define a [MASK] token/id.")

    def __call__(self, batch: List[Union[Dict, List[Dict]]]) -> Dict[str, torch.Tensor]:
        chunks = self._flatten(batch)
        out = self._stack(chunks)
        masked_ids, labels = self._mask_batch(out["input_ids"], out["attention_mask"])
        out["input_ids"] = masked_ids
        out["labels"] = labels
        return out


    def _flatten(self, batch) -> List[Dict]:
        out: List[Dict] = []
        for item in batch:
            if isinstance(item, dict):
                out.append(item)
            elif isinstance(item, (list, tuple)):
                out.extend(item)
            else:
                raise TypeError(f"Unexpected item type: {type(item)}")
        if not out:
            raise ValueError("Empty batch after normalization.")
        return out

    def _stack(self, chunks: List[Dict]) -> Dict[str, torch.Tensor]:
        keys = list(chunks[0].keys())
        out = {}
        for k in keys:
            # Skip known non-numeric or variable-shaped fields
            if k in ("text_values",):  # add others you don't want to collate
                continue

            # Replace Nones with safe defaults
            seq_list = []
            for c in chunks:
                v = c[k]
                if isinstance(v, list):
                    v = [0 if x is None else x for x in v]  # 0 for ints/floats
                elif v is None:
                    # single value case (shouldn't happen for sequences, but guard anyway)
                    v = 0
                seq_list.append(torch.as_tensor(v))
            out[k] = torch.stack(seq_list, 0)
        return out

    def _build_protected_ids(self, protected_tokens: List[str]) -> torch.BoolTensor:

        ids = set()
        for tok in protected_tokens:
            if tok in self.tokenizer.code2id:
                ids.add(self.tokenizer.code2id[tok])
        mask = torch.zeros(self.tokenizer.vocab_size, dtype=torch.bool)
        for i in ids:
            mask[i] = True
        return mask

    def _mask_batch(
        self,
        input_ids: torch.Tensor,      # (B, L)
        attention_mask: torch.Tensor  # (B, L)
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Vectorized BERT-style masking:
        • 80% -> [MASK]
        • 10% -> random token (not protected)
        • 10% -> keep original
        Returns masked_input_ids, labels.
        """
        device = input_ids.device
        prot = self.protected_ids.to(device)
        eligible = attention_mask.bool() & (~prot[input_ids])

        sample = torch.rand_like(input_ids.float()) < self.mask_prob
        to_mask = eligible & sample

        masked = input_ids.clone()
        labels = torch.full_like(input_ids, -100)
        labels[to_mask] = input_ids[to_mask]

        r = torch.rand_like(input_ids.float())
        to_mask80 = to_mask & (r < self.replace_prob)
        to_rand10 = to_mask & (r >= self.replace_prob) & (r < self.replace_prob + self.random_prob)

        # 80% -> [MASK]
        masked[to_mask80] = self.tokenizer.mask_id

        # 10% -> random allowed token
        allowed = (~prot).nonzero(as_tuple=False).squeeze(1).to(device)
        if allowed.numel() == 0:
            allowed = torch.arange(self.tokenizer.vocab_size, device=device)
        if to_rand10.any():
            rand_ids = allowed[torch.randint(0, allowed.numel(), (to_rand10.sum(),), device=device)]
            masked[to_rand10] = rand_ids
        return masked, labels

In [6]:
PROTECTED_TOKENS = [
    "[PAD]", "[CLS]", "[MASK]",
    "OUTPATIENT-START","OUTPATIENT-END",
    "EMERGENCY-START","EMERGENCY-END",
    "ADMISSION-AT-HOSPITAL","ADMISSION-AT-ICU",
    "DISCHARGE-FROM-HOSPITAL","DISCHARGE-FROM-ICU"]

limits = {
    'within24_query': {512:  ['w24_start_512',  'w24_end_512' ],
                       1024: ['w24_start_1024', 'w24_end_1024'],
                       1536: ['w24_start_1536', 'w24_end_1536'],
                      },
    
    'within24_hist_icu': {512: ['wStay_min', 'w24_start_512' ],
                         1024: ['wStay_min', 'w24_start_1024'],
                         1536: ['wStay_min', 'w24_start_1536'],
                      },
    
    'within24_hist_full': {512: [ 0, 'w24_start_512' ],
                          1024: [ 0, 'w24_start_1024'],
                          1536: [ 0, 'w24_start_1536'],
                          },

    
    'within48_query': {512:  ['w48_start_512',  'w48_end_512' ],
                       1024: ['w48_start_1024', 'w48_end_1024'],
                       1536: ['w48_start_1536', 'w48_end_1536'],
                      },

    'within48_hist_icu': {512: ['wStay_min', 'w48_start_512' ],
                         1024: ['wStay_min', 'w48_start_1024'],
                         1536: ['wStay_min', 'w48_start_1536']
                      },
    
    'within48_hist_full': {512: [ 0, 'w48_start_512' ],
                          1024: [ 0, 'w48_start_1024'],
                          1536: [ 0, 'w48_start_1536']
                          },
    
    'within_stay_query': {512:  ['wStay_start_512',  'wStay_end_512' ],
                          1024: ['wStay_start_1024', 'wStay_end_1024'],
                          1536: ['wStay_start_1536', 'wStay_end_1536']
                         },
    
    'within_stay_hist_icu': {512:  ['wStay_min', 'wStay_start_512' ],
                             1024: ['wStay_min', 'wStay_start_1024'],
                             1536: ['wStay_min', 'wStay_start_1536'],
                            },
    
    'within_stay_hist_full': {512:  [ 0, 'w48_start_512' ],
                              1024: [ 0, 'w48_start_1024'],
                              1536: [ 0, 'w48_start_1536'],
                             },
    }

In [7]:
class EvalDataset(Dataset):
    def __init__(self,
                 dataset_path: str,
                 data_idx_path: str,
                 seq_gen: SequencesGenerator,
                 limits_dict: dict,
                 task: str = 'y_mort',
                 main_window: str = 'within48_query', 
                 seq_length: int = 512,
                 use_time: bool = True,
                 use_numeric: bool = False,
                 split: str = 'train') -> None:
        
        needed_cols = ['subject_id', 'input_ids', 'attention_mask', 
                       'visit_ids', 'stage_ids', 'type_ids']

        if use_time:
            needed_cols.append('time_diff')
        if use_numeric:
            needed_cols.append('numeric_values')
            needed_cols.append('numeric_mask')
        self.start_limit = limits_dict[main_window][seq_length][0]
        self.end_limit   = limits_dict[main_window][seq_length][1]
        self.task = task
        
        self.seq_gen = seq_gen
        self.data_idx =  pl.scan_parquet(data_idx_path).collect()
        self.data_idx =  self.data_idx.filter(pl.col('split') == split)
        
        sub_ids = set(self.data_idx.get_column("subject_id").to_list())
        hf_dataset = load_from_disk(dataset_path)

        
        hf_dataset = hf_dataset.filter(
            lambda sids: [sid in sub_ids for sid in sids],
            batched=True,
            input_columns="subject_id",
        )

        # (then continue)
        self.hf_dataset = (
            hf_dataset
            .flatten_indices()
            .select_columns(needed_cols)
            .with_format("numpy", columns=needed_cols, output_all_columns=False)
        )
        
        
        sids = self.hf_dataset["subject_id"]        
        self.index = defaultdict(list)
        for i, sid in enumerate(sids):
            self.index[sid].append(i)
        

        
    def __len__(self) -> int:
        return len(self.data_idx)


    
    def __getitem__(self,
                    idx: int):
        
        stay = self.data_idx[idx]
        subject_id = stay['subject_id'][0]
        label = stay[self.task][0]
       
        
        
        start = stay[self.start_limit][0]
        end = stay[self.end_limit][0]

        
        timeline_encoded = self.hf_dataset.select(self.index[subject_id])[0]
        prediction_window = {k: (v[start:end] if isinstance(v, (list, np.ndarray)) else v) for k, v in timeline_encoded.items()}
        prediction_window = self.seq_gen.get_overlapped_chunks(prediction_window)
        prediction_window[0]['label'] = label
        
        return prediction_window[0]

In [8]:
class EvalCollator:
    def __init__(self) -> None:
        pass

    def __call__(self, batch: List[Union[Dict, List[Dict]]]) -> Dict[str, torch.Tensor]:
        chunks = self._flatten(batch)
        out = self._stack(chunks)

        # ---- CLEAN NUMERIC VALUES ----
        if "numeric_values" in out:
            vals = out["numeric_values"].float()          # [B, L]
            finite_mask = torch.isfinite(vals)            # True where not NaN/inf

            # if numeric_mask already exists, AND it with finite_mask
            if "numeric_mask" in out:
                mask = out["numeric_mask"].bool() & finite_mask
            else:
                mask = finite_mask

            # replace NaN/inf with 0.0 (or any neutral value)
            vals = torch.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)

            out["numeric_values"] = vals
            out["numeric_mask"] = mask

#         # ---- OPTIONAL: CLEAN TIME FEATURES TOO ----
#         if "time_diff" in out:
#             t = out["time_diff"].float()
#             t = torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)
#             out["time_diff"] = t

        return out

    def _flatten(self, batch) -> List[Dict]:
        out: List[Dict] = []
        for item in batch:
            if isinstance(item, dict):
                out.append(item)
            elif isinstance(item, (list, tuple)):
                out.extend(item)
            else:
                raise TypeError(f"Unexpected item type: {type(item)}")
        if not out:
            raise ValueError("Empty batch after normalization.")
        return out

    def _stack(self, chunks: List[Dict]) -> Dict[str, torch.Tensor]:
        keys = list(chunks[0].keys())
        out = {}
        for k in keys:
            # Skip known non-numeric or variable-shaped fields
            if k in ("text_values",):  # add others you don't want to collate
                continue

            seq_list = []
            for c in chunks:
                v = c[k]
                if isinstance(v, list):
                    v = [0 if x is None else x for x in v]
                elif v is None:
                    v = 0
                seq_list.append(torch.as_tensor(v))
            out[k] = torch.stack(seq_list, 0)
        return out

In [9]:
import os
import torch


import lightning as lt
import torch.nn as nn
from transformers import RoFormerModel, RoFormerForMaskedLM
from torchmetrics.classification import Accuracy, BinaryAUROC, BinaryAveragePrecision

In [10]:
class Time2Vec(nn.Module):

    def __init__(
        self,
        in_features: int = 1,
        out_features: int = 16,
        periodic_activation: Callable = torch.sin,
    ):
        super().__init__()
        assert out_features >= 1, "out_features must be >= 1"

        self.in_features = in_features
        self.out_features = out_features
        self.periodic_activation = periodic_activation

        self.W = nn.Parameter(torch.randn(in_features, out_features - 1))
        self.b = nn.Parameter(torch.randn(out_features - 1))

        self.W0 = nn.Parameter(torch.randn(in_features))
        self.b0 = nn.Parameter(torch.randn(1))

    def forward(self, tau: torch.Tensor) -> torch.Tensor:

        v1 = self.periodic_activation(tau @ self.W + self.b)
        v2 = (tau @ self.W0).unsqueeze(-1) + self.b0

        return torch.cat([v2, v1], dim=-1)
    

In [11]:
class EHREmbeddings(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_size: int,
        pad_token_id: int = 0,
        type_vocab_size: int = 28,
        visit_vocab_size: int = 102,
        stage_vocab_size: int = 5,
        dropout: float = 0.1,
        use_position_embeddings: bool = False,
        max_position_embeddings: int = 0,
        use_time: bool = True,
        time_in_features: int = 1,
        time_out_features: int = 16,
        use_numeric: bool = True,
        numeric_hidden_size: int = 16,   # <-- small bottleneck for numeric
    ):
        super().__init__()

        self.tok_emb   = nn.Embedding(vocab_size,       embedding_size, padding_idx=pad_token_id)
        self.type_emb  = nn.Embedding(type_vocab_size,  embedding_size, padding_idx=pad_token_id)
        self.visit_emb = nn.Embedding(visit_vocab_size, embedding_size, padding_idx=pad_token_id)
        self.stage_emb = nn.Embedding(stage_vocab_size, embedding_size, padding_idx=pad_token_id)

        # ---- positional (local) ----
        self.use_position_embeddings = use_position_embeddings
        if use_position_embeddings:
            if max_position_embeddings <= 0:
                raise ValueError("max_position_embeddings must be > 0 when use_position_embeddings=True")
            self.pos_emb = nn.Embedding(max_position_embeddings, embedding_size)
        else:
            self.pos_emb = None

        # ---- time (Time2Vec) ----
        self.use_time = use_time
        if use_time:
            self.time2vec = Time2Vec(
                in_features=time_in_features,
                out_features=time_out_features,
                periodic_activation=torch.sin,
            )
            self.time_proj = nn.Linear(time_out_features, embedding_size)
        else:
            self.time2vec = None
            self.time_proj = None

        # ---- numeric values ----
        self.use_numeric = use_numeric
        if use_numeric:
            self.numeric_hidden_size = numeric_hidden_size
            # 1 scalar -> small hidden -> embedding_size
            self.num_proj1 = nn.Linear(1, numeric_hidden_size)
            self.num_proj2 = nn.Linear(numeric_hidden_size, embedding_size)
            self.num_act = nn.GELU()

            # learned embedding for "no numeric value"
            self.null_numeric = nn.Parameter(torch.zeros(embedding_size))
            nn.init.normal_(self.null_numeric, mean=0.0, std=0.02)

            nn.init.xavier_uniform_(self.num_proj1.weight)
            nn.init.zeros_(self.num_proj1.bias)
            nn.init.xavier_uniform_(self.num_proj2.weight)
            nn.init.zeros_(self.num_proj2.bias)
        else:
            self.num_proj1 = None
            self.num_proj2 = None
            self.num_act = None
            self.null_numeric = None

        self.norm = nn.LayerNorm(embedding_size)
        self.drop = nn.Dropout(dropout)

    def encode(
        self,
        input_ids,
        type_ids,
        visit_ids,
        stage_ids,
        time_feats=None,          # (B, L) or (B, L, time_in_features)
        numeric_values=None,      # (B, L) normalized in [-3, 3]
        numeric_mask=None,        # (B, L) bool/int: True if numeric is present
    ):
        # base token + type + visit + stage
        x = self.tok_emb(input_ids.long())
        x = x + self.type_emb(type_ids.long())
        x = x + self.visit_emb(visit_ids.long())
        x = x + self.stage_emb(stage_ids.long())

        # positional (local) embeddings
        if self.pos_emb is not None:
            bsz, seqlen = input_ids.size()
            position_ids = torch.arange(
                seqlen, device=input_ids.device
            ).unsqueeze(0).expand(bsz, seqlen)
            x = x + self.pos_emb(position_ids)

        # time (Time2Vec)
        if self.use_time:
            if time_feats is None:
                raise ValueError("time_feats must be provided when use_time=True")
            if time_feats.dim() == 2:
                time_feats = time_feats.unsqueeze(-1)
            elif time_feats.dim() != 3:
                raise ValueError(f"Unexpected time_feats.dim()={time_feats.dim()}, expected 2 or 3")
            t = self.time2vec(time_feats.float())   # (B, L, time_out_features)
            t = self.time_proj(t)                   # (B, L, embedding_size)
            x = x + t

        # numeric values
        if self.use_numeric:
            if numeric_values is None or numeric_mask is None:
                raise ValueError("numeric_values and numeric_mask must be provided when use_numeric=True")

            # (optional safety) clamp extreme values
            v = numeric_values.float().unsqueeze(-1)        # (B, L, 1)
            # small bottleneck then project to emb size
            h = self.num_act(self.num_proj1(v))             # (B, L, H_num)
            num_emb = self.num_proj2(h)                     # (B, L, D)

            mask = numeric_mask.bool().unsqueeze(-1)        # (B, L, 1)
            num_emb = torch.where(mask, num_emb, self.null_numeric.view(1, 1, -1))
            x = x + num_emb

        return self.drop(self.norm(x))

    def forward(self, input_ids=None, token_type_ids=None, inputs_embeds=None, **kwargs):
        if inputs_embeds is not None:
            return inputs_embeds
        x = self.tok_emb(input_ids.long())
        return self.drop(self.norm(x))

In [12]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union, cast

import torch
import torch.nn.functional as F
from torch import nn
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss
from transformers.activations import ACT2FN
from transformers.models.mamba.modeling_mamba import (
    MambaModel,
    MambaPreTrainedModel,
)
from transformers.utils import (
    ModelOutput,
    add_start_docstrings,
    add_start_docstrings_to_model_forward,
    replace_return_docstrings,
)







@dataclass
class MambaSequenceClassifierOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: Optional[torch.FloatTensor] = None  # Make optional to allow None default
    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None



class MambaClassificationHead(nn.Module):
    """Head for sentence-level classification tasks."""

    def __init__(self, config: Any) -> None:
        """Initialize the head."""
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.classifier_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)
        self.config = config

    def forward(self, features: torch.Tensor, **kwargs: Any) -> torch.Tensor:
        """Forward pass."""
        x = features  # Pooling is done by the forward pass
        x = self.dropout(x)
        x = self.dense(x)
        x = ACT2FN[self.config.hidden_act](x)
        x = self.dropout(x)

        # Ensure we return a proper torch.Tensor
        result = self.out_proj(x)
        return torch.as_tensor(result, dtype=torch.float32)

class MambaForSequenceClassification(MambaPreTrainedModel):  # type: ignore
    def __init__(self, config: Any) -> None:
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        self.backbone = MambaModel(config)
        self.classifier = MambaClassificationHead(config)

        # Initialize weights and apply final processing
        self.post_init()


    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[MambaSequenceClassifierOutput, Tuple[torch.FloatTensor, ...]]:

        if inputs_embeds is not None:
            sequence_outputs = self.backbone(
                input_ids=None,
                inputs_embeds=inputs_embeds,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )
        else:
            sequence_outputs = self.backbone(
                input_ids=input_ids,
                inputs_embeds=None,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )
        last_hidden_states = sequence_outputs[0]
        batch_size = last_hidden_states.shape[0]

        # Pool the hidden states for the last tokens before padding
        # to use for classification
        if input_ids is not None:
            # Cast input_ids to Tensor for torch.eq to work properly
            input_ids_tensor = torch.as_tensor(input_ids, device=last_hidden_states.device)
            last_token_indexes = (
                torch.eq(input_ids_tensor, self.config.pad_token_id).int().argmax(-1) - 1
            )
        else:
            # Use default indices if input_ids is None
            last_token_indexes = torch.zeros(batch_size, dtype=torch.long, device=last_hidden_states.device)
        # Convert last_token_indexes to tensor if needed
        last_token_indexes_tensor = torch.as_tensor(last_token_indexes, device=last_hidden_states.device)
        pooled_last_hidden_states = last_hidden_states[
            torch.arange(batch_size, device=last_hidden_states.device),
            last_token_indexes_tensor,
        ]

        logits = self.classifier(pooled_last_hidden_states)

        loss = None
        if labels is not None:
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (labels.dtype in [torch.long, torch.int]):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct_regression = MSELoss()
                if self.num_labels == 1:
                    loss = loss_fct_regression(logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct_regression(logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct_classification = CrossEntropyLoss()
                loss = loss_fct_classification(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct_multilabel = BCEWithLogitsLoss()
                loss = loss_fct_multilabel(logits, labels)

        if not return_dict:
            output = (logits,) + sequence_outputs[1:]
            return ((loss,) + output) if loss is not None else output  # type: ignore

        # Type cast loss and logits to ensure correct types for MambaSequenceClassifierOutput
        float_loss: Optional[torch.FloatTensor] = None
        if isinstance(loss, torch.Tensor):
            float_loss = cast(torch.FloatTensor, loss.to(dtype=torch.float32))

        # Cast logits to FloatTensor
        float_logits = cast(torch.FloatTensor, torch.as_tensor(logits, dtype=torch.float32))

        return MambaSequenceClassifierOutput(
            loss=float_loss,
            logits=float_logits,
            hidden_states=sequence_outputs.hidden_states,
        )

In [13]:
import pytorch_lightning as lt
import torch
from torch import nn
from torch.cuda.amp import autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, SequentialLR
from transformers import MambaConfig
from transformers.models.mamba.modeling_mamba import (
    MambaCausalLMOutput,
    MambaForCausalLM,
)



class MambaPretrain(lt.LightningModule):

    def __init__(
        self,
        vocab_size: int,
        embedding_size: int = 768,
        type_vocab_size: int = 28,
        visit_vocab_size: int = 102,
        stage_vocab_size: int = 5,
        max_seq_length: int = 2048,
        state_size: int = 16,
        num_hidden_layers: int = 32,
        expand: int = 2,
        conv_kernel: int = 4,
        learning_rate: float = 5e-5,
        dropout_prob: float = 0.1,
        padding_idx: int = 0,
        cls_idx: int = 5,
        use_mambapy: bool = False,
        use_position_embeddings: bool = False,
        use_time: bool = True,
        time_in_features: int = 1,
        time_out_features: int = 16,
        use_numeric: bool = True,
        numeric_hidden_size: int = 16,
    ):
        super().__init__()

        self.save_hyperparameters()

        self.vocab_size = vocab_size
        self.embedding_size = embedding_size
        self.learning_rate = learning_rate
        self.padding_idx = padding_idx
        self.cls_idx = cls_idx

        # --- Mamba config (backbone) ---
        self.config = MambaConfig(
            vocab_size=self.vocab_size,
            hidden_size=self.embedding_size,
            state_size=state_size,
            num_hidden_layers=num_hidden_layers,
            expand=expand,
            conv_kernel=conv_kernel,
            pad_token_id=self.padding_idx,
            bos_token_id=self.cls_idx,
            eos_token_id=self.padding_idx,
            use_mambapy=use_mambapy,
        )

        # --- Saeed's EHR embeddings ---
        self.ehr_embeddings = EHREmbeddings(
            vocab_size=vocab_size,
            embedding_size=embedding_size,
            pad_token_id=padding_idx,
            type_vocab_size=type_vocab_size,
            visit_vocab_size=visit_vocab_size,
            stage_vocab_size=stage_vocab_size,
            dropout=dropout_prob,
            use_position_embeddings=use_position_embeddings,
            max_position_embeddings=max_seq_length if use_position_embeddings else 0,
            use_time=use_time,
            time_in_features=time_in_features,
            time_out_features=time_out_features,
            use_numeric=use_numeric,
            numeric_hidden_size=numeric_hidden_size,
        )

        # --- Mamba causal LM head ---
        self.model = MambaForCausalLM(config=self.config)

        # Use their standard initialization for all *new* modules we add
        self.post_init()

    # weight init copied from their style
    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def post_init(self) -> None:
        self.apply(self._init_weights)

    def forward(
        self,
        batch: dict,
        labels: torch.Tensor = None,
        output_hidden_states: bool  = False,
        return_dict: bool  = True,
    ) -> MambaCausalLMOutput:
        """
        batch is expected to contain:
            - input_ids       : (B, L)
            - type_ids        : (B, L)
            - visit_ids       : (B, L)
            - stage_ids       : (B, L)
            - attention_mask  : (B, L)
            - time_feats      : (B, L) or (B, L, T)     [optional if use_time=True]
            - numeric_values  : (B, L)                  [optional if use_numeric=True]
            - numeric_mask    : (B, L) bool/int         [optional if use_numeric=True]
        """
        input_ids = batch["input_ids"]
        type_ids = batch["type_ids"]
        visit_ids = batch["visit_ids"]
        stage_ids = batch["stage_ids"]
        attention_mask = batch.get("attention_mask", (input_ids != self.padding_idx).long())

        time_feats = batch.get("time_feats", None)
        numeric_values = batch.get("numeric_values", None)
        numeric_mask = batch.get("numeric_mask", None)

        inputs_embeds = self.ehr_embeddings.encode(
            input_ids=input_ids,
            type_ids=type_ids,
            visit_ids=visit_ids,
            stage_ids=stage_ids,
            time_feats=time_feats,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
        )

        if labels is None:
            labels = batch.get("labels", None)
            if labels is None:
                # By default, causal LM uses input_ids as labels.
                labels = input_ids

        return self.model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

    def training_step(self, batch: dict, batch_idx: int):
        with autocast():
            out = self.forward(batch, return_dict=True)
            loss = out.loss

        (current_lr,) = self.lr_schedulers().get_last_lr()
        self.log_dict(
            {"train_loss": loss, "lr": current_lr},
            on_step=True,
            prog_bar=True,
            sync_dist=True,
        )
        return loss

    def validation_step(self, batch: dict, batch_idx: int):
        with autocast():
            out = self.forward(batch, return_dict=True)
            loss = out.loss

        (current_lr,) = self.lr_schedulers().get_last_lr()
        self.log_dict(
            {"val_loss": loss, "lr": current_lr},
            on_step=True,
            prog_bar=True,
            sync_dist=True,
        )
        return loss

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.learning_rate)

        n_steps = self.trainer.estimated_stepping_batches
        n_warmup_steps = int(0.1 * n_steps)
        n_decay_steps = int(0.9 * n_steps)

        warmup = LinearLR(
            optimizer,
            start_factor=0.01,
            end_factor=1.0,
            total_iters=n_warmup_steps,
        )
        decay = LinearLR(
            optimizer,
            start_factor=1.0,
            end_factor=0.01,
            total_iters=n_decay_steps,
        )
        scheduler = SequentialLR(
            optimizer=optimizer,
            schedulers=[warmup, decay],
            milestones=[n_warmup_steps],
        )

        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]

In [14]:
from typing import List, Dict, Any, Union, Tuple
import torch

class CausalDataCollator:
    def __call__(self, batch: List[Union[Dict, List[Dict]]]) -> Dict[str, torch.Tensor]:
        # flatten nested lists (same pattern as your other collators)
        chunks: List[Dict[str, Any]] = []
        for item in batch:
            if isinstance(item, dict):
                chunks.append(item)
            elif isinstance(item, (list, tuple)):
                chunks.extend(item)
            else:
                raise TypeError(f"Unexpected item type: {type(item)}")

        if not chunks:
            raise ValueError("Empty batch after normalization.")

        keys = list(chunks[0].keys())
        out: Dict[str, torch.Tensor] = {}

        for k in keys:
            # skip text if present
            if k in ("text_values",):
                continue

            seq_list = []
            for c in chunks:
                v = c[k]
                if isinstance(v, list):
                    v = [0 if x is None else x for x in v]
                elif v is None:
                    v = 0
                seq_list.append(torch.as_tensor(v))
            out[k] = torch.stack(seq_list, 0)

        # labels = input_ids, ignore padding in loss
        input_ids = out["input_ids"]
        attention_mask = out["attention_mask"]
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        out["labels"] = labels
        return out

In [15]:
import pytorch_lightning as lt
import torch
from torch import nn
from torch.cuda.amp import autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, SequentialLR
from torchmetrics.classification import Accuracy, F1Score, AUROC, Precision, Recall

class MambaFinetune(lt.LightningModule):


    def __init__(
        self,
        pretrained_model: MambaPretrain,
        problem_type: str = "single_label_classification",
        num_labels: int = 2,
        learning_rate: float = 5e-5,
        classifier_dropout: float = 0.1,
    ):
        super().__init__()

        self.num_labels = num_labels
        self.learning_rate = learning_rate
        self.classifier_dropout = classifier_dropout

        # --- config from pretrained ---
        self.config = pretrained_model.config
        self.config.num_labels = self.num_labels
        self.config.classifier_dropout = self.classifier_dropout
        self.config.problem_type = problem_type

        # --- backbone + classifier ---
        self.model = MambaForSequenceClassification(config=self.config)

        # reuse pretrained backbone
        self.pretrained_model = pretrained_model
        self.ehr_embeddings = self.pretrained_model.ehr_embeddings
        self.model.backbone = self.pretrained_model.model.backbone

        # --- metrics (torchmetrics) ---
        # choose task type dynamically
        if problem_type == "multi_label_classification":
            task_mode = "multilabel"
        else:
            if num_labels == 2:
                task_mode = "binary"
            else:
                task_mode = "multiclass"

        self.train_acc = Accuracy(task=task_mode, num_labels=num_labels if task_mode != "binary" else None)
        self.val_acc = Accuracy(task=task_mode, num_labels=num_labels if task_mode != "binary" else None)
        self.test_acc = Accuracy(task=task_mode, num_labels=num_labels if task_mode != "binary" else None)

        self.test_f1 = F1Score(task=task_mode, num_labels=num_labels if task_mode != "binary" else None)
        self.test_auc = AUROC(task=task_mode, num_labels=num_labels if task_mode != "binary" else None)
        self.test_precision = Precision(task=task_mode, num_labels=num_labels if task_mode != "binary" else None)
        self.test_recall = Recall(task=task_mode, num_labels=num_labels if task_mode != "binary" else None)

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def post_init(self) -> None:
        self.apply(self._init_weights)

    def _encode_batch(self, batch: dict) -> torch.Tensor:
        """
        Common embedding logic: use EHREmbeddings on the batch.
        """
        input_ids = batch["input_ids"]
        type_ids = batch["type_ids"]
        visit_ids = batch["visit_ids"]
        stage_ids = batch["stage_ids"]

        time_feats = batch.get("time_feats", None)
        numeric_values = batch.get("numeric_values", None)
        numeric_mask = batch.get("numeric_mask", None)

        inputs_embeds = self.ehr_embeddings.encode(
            input_ids=input_ids,
            type_ids=type_ids,
            visit_ids=visit_ids,
            stage_ids=stage_ids,
            time_feats=time_feats,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
        )
        return inputs_embeds

    def forward(
        self,
        batch: dict,
        labels: torch.Tensor= None,
        output_hidden_states: bool =False,
        return_dict: bool =True,
    ) -> MambaSequenceClassifierOutput:
        """
        batch is expected to contain:
            - input_ids       : (B, L)
            - type_ids        : (B, L)
            - visit_ids       : (B, L)
            - stage_ids       : (B, L)
            - attention_mask  : (B, L)
            - [optionally time_feats, numeric_values, numeric_mask]
            - labels          : (B,)
        """
        input_ids = batch["input_ids"]
        attention_mask = batch.get("attention_mask", (input_ids != self.config.pad_token_id).long())

        if labels is None:
            labels = batch.get("labels", None)

        inputs_embeds = self._encode_batch(batch)

        return self.model(
            input_ids=input_ids,
            inputs_embeds=inputs_embeds,
            labels=labels,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

    # -------- training / validation ----------

    def training_step(self, batch: dict, batch_idx: int):
        labels = batch["labels"]

        with autocast():
            outputs = self.forward(batch, labels=labels, return_dict=True)
            loss = outputs.loss
            logits = outputs.logits

        preds = torch.argmax(logits, dim=-1)

        self.train_acc.update(preds, labels)

        (current_lr,) = self.lr_schedulers().get_last_lr()
        self.log("train_loss", loss, on_step=True, prog_bar=True, sync_dist=True)
        self.log("train_acc", self.train_acc, on_step=True, prog_bar=True, sync_dist=True)
        self.log("lr", current_lr, on_step=True, prog_bar=False, sync_dist=True)

        return loss

    def validation_step(self, batch: dict, batch_idx: int):
        labels = batch["labels"]

        with autocast():
            outputs = self.forward(batch, labels=labels, return_dict=True)
            loss = outputs.loss
            logits = outputs.logits

        preds = torch.argmax(logits, dim=-1)
        self.val_acc.update(preds, labels)

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("val_acc", self.val_acc, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)

        return loss

    # -------- test ----------

    def test_step(self, batch: dict, batch_idx: int):
        labels = batch["labels"]

        with autocast():
            outputs = self.forward(batch, labels=labels, return_dict=True)
            loss = outputs.loss
            logits = outputs.logits

        preds = torch.argmax(logits, dim=-1)
        probs = torch.softmax(logits, dim=-1)

        self.test_acc.update(preds, labels)
        self.test_f1.update(preds, labels)
        # AUROC expects prob for positive class (binary) or probs (multiclass)
        if self.num_labels == 2:
            pos_probs = probs[:, 1]
            self.test_auc.update(pos_probs, labels)
        else:
            self.test_auc.update(probs, labels)
        self.test_precision.update(preds, labels)
        self.test_recall.update(preds, labels)

        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)

        return {"loss": loss}

    def on_test_epoch_end(self):
        self.log("test_acc", self.test_acc.compute(), prog_bar=True, sync_dist=True)
        self.log("test_f1", self.test_f1.compute(), prog_bar=True, sync_dist=True)
        self.log("test_auc", self.test_auc.compute(), prog_bar=True, sync_dist=True)
        self.log("test_precision", self.test_precision.compute(), prog_bar=False, sync_dist=True)
        self.log("test_recall", self.test_recall.compute(), prog_bar=False, sync_dist=True)

    # -------- optim ----------

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.learning_rate)

        n_steps = self.trainer.estimated_stepping_batches
        n_warmup_steps = int(0.1 * n_steps)
        n_decay_steps = int(0.9 * n_steps)

        warmup = LinearLR(
            optimizer,
            start_factor=0.01,
            end_factor=1.0,
            total_iters=n_warmup_steps,
        )
        decay = LinearLR(
            optimizer,
            start_factor=1.0,
            end_factor=0.01,
            total_iters=n_decay_steps,
        )
        scheduler = SequentialLR(
            optimizer=optimizer,
            schedulers=[warmup, decay],
            milestones=[n_warmup_steps],
        )

        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]

In [16]:
from torch.utils.data import DataLoader
from pytorch_lightning import Trainer
import polars as pl


tokenizer_path   = "../vocab.json"      # where you saved Tokenizer
dataset_path     = "../data/meds_normalized_arrow/" # load_from_disk path
data_idx_path    = "../pretrain_idx.parquet"

# 1) sequence generator (only used for indexing in this version, but keep it consistent)
seq_gen = SequencesGenerator(
    tokenizer_path=tokenizer_path,
    chunk_length=512,
    overlap=128,
    return_numeric=False,  # or True if your HF dataset has numeric stream integrated
    return_text=False,
    return_time=False,     # or True if using time_diff
    return_ids=False,
)

# 2) dataset instance
train_dataset = EHRPretrainDataset(
    dataset_path=dataset_path,
    data_idx_path=data_idx_path,
    seq_generator=seq_gen,
    needed_cols=['subject_id', 'input_ids', 'attention_mask', 'visit_ids', 'stage_ids', 'type_ids'],
    split="train",
)

# val_dataset = EHRPretrainDataset(
#     dataset_path=dataset_path,
#     data_idx_path=data_idx_path,
#     seq_generator=seq_gen,
#     needed_cols=['subject_id', 'input_ids', 'attention_mask', 'visit_ids', 'stage_ids', 'type_ids'],
#     split="val",
# )

# 3) collator + dataloaders
causal_collator = CausalDataCollator()

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    collate_fn=causal_collator,
)

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=8,
#     shuffle=False,
#     num_workers=4,
#     collate_fn=causal_collator,
# )



mamba_pretrain = MambaPretrain(
    vocab_size=seq_gen.tokenizer.vocab_size,
    embedding_size=768,
    type_vocab_size=len(seq_gen.tokenizer.type2id),
    visit_vocab_size=102,          # match EHREmbeddings
    stage_vocab_size=5,            # match EHREmbeddings
    max_seq_length=512,           # or whatever chunk length you use
    state_size=16,
    num_hidden_layers=32,
    expand=2,
    conv_kernel=4,
    learning_rate=1e-6,
    dropout_prob=0.1,
    padding_idx=seq_gen.tokenizer.pad_id or 0,   # <-- renamed
    cls_idx=seq_gen.tokenizer.cls_id or 0,
    use_mambapy=True,
    use_position_embeddings=False,  # or True if you enabled it in EHREmbeddings
    use_time=False,
    time_in_features=1,
    time_out_features=16,
    use_numeric=False,              # or True if your dataset returns numeric_values/mask
    numeric_hidden_size=16,
)

# 5) trainer just to test everything runs
trainer_pretrain = Trainer(
    max_epochs=1,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed",
#     accumulate_grad_batches=1, # you can increase this later if needed

)

# sanity run
trainer_pretrain.fit(
    mamba_pretrain,
    train_dataloaders=train_loader,
#     val_dataloaders=val_loader,
)

Loading dataset from disk:   0%|          | 0/70 [00:00<?, ?it/s]

The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the mamba.py backend. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d
/home/sas10092/.conda/envs/med-ehr/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/sas10092/.conda/envs/med-ehr/lib/python3.9/sit ...
Using 16bit Automatic Mixed Precision (AMP)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/sas10092/.con

Training: |          | 0/? [00:00<?, ?it/s]

/tmpdata/ipykernel_2915551/3329078114.py:159: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():

Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [17]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (build):", torch.version.cuda)
print("Current device:", torch.cuda.current_device() if torch.cuda.is_available() else None)
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

PyTorch version: 2.7.0+cu126
CUDA available: True
CUDA version (build): 12.6
Current device: 0
Device name: NVIDIA A100 80GB PCIe


In [25]:
!module avail cuda 

--------------------- /share/apps/NYUAD5/modules/SOFTWARE ----------------------
cuda/10.0.130  cuda/11.2.67  cuda/11.8.0  cuda/12.2.0  
>

In [26]:
!module load cuda/12.2.0

Loading module 'cuda/12.2.0'
Loading module 'gcc/9.2.0'

Loading cuda/12.2.0
  Loading requirement: gcc/9.2.0
>

In [27]:
!nvcc --version

/bin/bash: nvcc: command not found


In [1]:
import torch

# Check mamba-ssm
import mamba_ssm
print("mamba_ssm version:", getattr(mamba_ssm, "__version__", "no __version__"))

# These are the important ops
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn, selective_state_update

print("selective_scan_fn is None?       ", selective_scan_fn is None)
print("selective_state_update is None?  ", selective_state_update is None)

# Check causal-conv1d
from causal_conv1d import causal_conv1d_fn, causal_conv1d_update

print("causal_conv1d_fn is None?        ", causal_conv1d_fn is None)
print("causal_conv1d_update is None?    ", causal_conv1d_update is None)

ImportError: /lib64/libc.so.6: version `GLIBC_2.32' not found (required by /home/sas10092/.conda/envs/med-ehr/lib/python3.9/site-packages/selective_scan_cuda.cpython-39-x86_64-linux-gnu.so)

In [2]:
!ldd --version | head -1

ldd (GNU libc) 2.28


In [3]:
!nvcc --version

/bin/bash: nvcc: command not found


In [4]:
import torch
print("torch:", torch.__version__)

try:
    import mamba_ssm
    print("mamba_ssm:", getattr(mamba_ssm, "__version__", "no __version__"))
except Exception as e:
    print("mamba_ssm import error:", repr(e))

torch: 2.7.0+cu126
mamba_ssm import error: ModuleNotFoundError("No module named 'mamba_ssm'")
